<a href="https://colab.research.google.com/github/reyho0n/SQL-challanges/blob/main/Content_Prioritization_and_translation_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1- Installing dependencies

In [ ]:
!pip install -q google-api-python-client google-auth google-auth-oauthlib \
                requests beautifulsoup4 readability-lxml openai


2- Import Libraries

In [ ]:
import os
import json
import base64
import shutil
import requests
import pandas as pd
from datetime import date, timedelta
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from readability import Document
from googleapiclient.discovery import build
from google.oauth2 import service_account
from google.colab import drive
from openai import OpenAI




3- Load Secrets and config

In [ ]:
import google.colab.userdata

OPENAI_API_KEY   = google.colab.userdata.get('OPENAI_API_KEY')
GSC_SITE_URL     = google.colab.userdata.get('GSC_SITE_URL')
WP_USER          = google.colab.userdata.get('WP_USER')
WP_APP_PASSWORD  = google.colab.userdata.get('WP_APP_PASSWORD')
WP_URL           = "https://www.helpling.de"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"✅ Secrets loaded | WordPress target: {WP_URL}")




✅ Secrets loaded | WordPress target: https://www.helpling.de


4- Mount Google Drive

In [ ]:
drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive/EN Blog Drafts"
LOG_FILE     = f"{DRIVE_FOLDER}/translation_log.csv"

os.makedirs(DRIVE_FOLDER, exist_ok=True)
print(f"✅ Drive mounted | Output folder: {DRIVE_FOLDER}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted | Output folder: /content/drive/MyDrive/EN Blog Drafts


5- Google Service Auth _ GSC





In [ ]:
SA_FILE_IN_DRIVE = f"{DRIVE_FOLDER}/en-content-ai-agent-159dfe946492.json"
SA_FILE_LOCAL    = "en-content-ai-agent-159dfe946492.json"

if not os.path.exists(SA_FILE_LOCAL):
    shutil.copy(SA_FILE_IN_DRIVE, SA_FILE_LOCAL)
    print("✅ Service account file copied from Drive")

SCOPES = [
    "https://www.googleapis.com/auth/webmasters.readonly",
    "https://www.googleapis.com/auth/documents",
]

credentials = service_account.Credentials.from_service_account_file(
    SA_FILE_LOCAL, scopes=SCOPES
)
gsc_service  = build("searchconsole", "v1", credentials=credentials)
docs_service = build("docs", "v1", credentials=credentials)

print("✅ GSC ready | Google Docs ready")



✅ Service account file copied from Drive
✅ GSC ready | Google Docs ready


6- Log Helpers

In [ ]:
def load_translated_urls():
    if os.path.exists(LOG_FILE):
        return set(pd.read_csv(LOG_FILE)["de_url"].tolist())
    return set()

def save_to_log(de_url, en_title, file_path, clicks, impressions, position, wp_id=None):
    new_row = pd.DataFrame([{
        "de_url":          de_url,
        "en_title":        en_title,
        "file_path":       file_path,
        "wp_draft_id":     wp_id,
        "clicks":          clicks,
        "impressions":     impressions,
        "position":        position,
        "translated_date": date.today().isoformat()
    }])
    if os.path.exists(LOG_FILE):
        df = pd.concat([pd.read_csv(LOG_FILE), new_row], ignore_index=True)
    else:
        df = new_row
    df.to_csv(LOG_FILE, index=False)

def view_log():
    if not os.path.exists(LOG_FILE):
        print("📋 No log yet.")
        return
    df = pd.read_csv(LOG_FILE)
    print(f"📋 Total translated: {len(df)}\n")
    print(df[["translated_date", "clicks", "en_title", "wp_draft_id", "de_url"]].to_string(index=False))

def remove_from_log(urls_to_remove):
    if not os.path.exists(LOG_FILE):
        print("❌ No log file found.")
        return
    df = pd.read_csv(LOG_FILE)
    before = len(df)
    df = df[~df["de_url"].isin(urls_to_remove)]
    df.to_csv(LOG_FILE, index=False)
    print(f"✅ Removed {before - len(df)} URL(s). Remaining: {len(df)}")

def clear_log():
    if os.path.exists(LOG_FILE):
        os.remove(LOG_FILE)
        print("✅ Log cleared.")

already_translated = load_translated_urls()
print(f"✅ Log loaded — {len(already_translated)} articles already translated")

✅ Log loaded — 2 articles already translated


7- GSC: Fetch Top German Blog Posts

In [ ]:
def get_top_german_blog_posts(gsc_service, site_url, days=30, row_limit=500):
    today = date.today()
    start = today - timedelta(days=days)

    response = gsc_service.searchanalytics().query(
        siteUrl=site_url,
        body={
            "startDate":  start.isoformat(),
            "endDate":    today.isoformat(),
            "dimensions": ["page"],
            "rowLimit":   row_limit,
            "searchType": "web"
        }
    ).execute()

    posts = []
    for row in response.get("rows", []):
        url = row["keys"][0]
        if "/blog/" in url and "/de_en/" not in url:
            posts.append({
                "url":         url,
                "clicks":      row.get("clicks", 0),
                "impressions": row.get("impressions", 0),
                "position":    round(row.get("position", 0), 1)
            })
    return posts

all_posts = get_top_german_blog_posts(gsc_service, GSC_SITE_URL)
print(f"✅ Found {len(all_posts)} German blog posts (last 30 days)\n")
print(f"{'Clicks':>8}  {'Impr.':>8}  {'Pos.':>6}  URL")
print("-" * 80)
for p in all_posts[:20]:
    print(f"{p['clicks']:>8.0f}  {p['impressions']:>8.0f}  {p['position']:>6}  {p['url']}")


✅ Found 197 German blog posts (last 30 days)

  Clicks     Impr.    Pos.  URL
--------------------------------------------------------------------------------
     726     66927     6.0  https://www.helpling.de/blog/silber-putzen-mit-alufolie-so-gehts/
     668     87057     4.8  https://www.helpling.de/blog/stundenlohn-putzfrau-in-deutschland/
     658     79678     2.6  https://www.helpling.de/blog/vergilbtes-plastik-reinigen-so-gehts-in-3-schritten/
     552     61019     3.2  https://www.helpling.de/blog/putzfrau-kosten-2026-in-deutschland/
     509     74053     5.7  https://www.helpling.de/blog/toilette-verstopft/
     398     44270     3.4  https://www.helpling.de/blog/wandfarbe-aus-kleidung-entfernen-so-klappts/
     386     33071     3.0  https://www.helpling.de/blog/fenster-putzen-mit-klarspueler-der-geheimtipp/
     260     25471     3.8  https://www.helpling.de/blog/massage-schwangerschaft-alles-was-sie-darueber-wissen-muessen/
     252     23820     8.2  https://www.helpli

8- Content Extraction, Translation & WordPress Helpers


In [ ]:
client = OpenAI()

def get_wp_headers():
    token = base64.b64encode(f"{WP_USER}:{WP_APP_PASSWORD}".encode()).decode("utf-8")
    return {"Authorization": f"Basic {token}", "Content-Type": "application/json"}

def fetch_article_html(url):
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    doc  = Document(resp.text)
    soup = BeautifulSoup(doc.summary(html_partial=True), "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return doc.title(), str(soup)

def translate_article_html(title_de, html_de):
    system_msg = (
           "You are a native English copywriter and SEO content specialist. "
        "Your task is to transcreate (not just translate) a German blog post into natural, engaging English. "
        "\n\n"
        "Rules:\n"
        "- Write as a native English speaker would — avoid word-for-word translation\n"
        "- Replace stiff or literal German phrasing with natural English expressions\n"
        "- Keep the same meaning, facts, and structure — but rewrite sentences that sound unnatural\n"
        "- Use active voice wherever possible\n"
        "- Vary sentence length to create rhythm — mix short punchy sentences with longer ones\n"
        "- For lifestyle/home content: warm, practical, and friendly tone — like a helpful friend, not a manual\n"
        "- The target audience is expats living in Germany — keep all facts, prices, and references German\n"
        "- Output only valid HTML, no explanations or meta-commentary\n"
        "\n"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.4,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": f"<h1>{title_de}</h1>\n{html_de}"}
        ]
    )
    return response.choices[0].message.content.strip()

def extract_title_and_body(translated_html, fallback_title):
    soup = BeautifulSoup(translated_html, "html.parser")
    h1   = soup.find("h1")
    title_en = h1.get_text(strip=True) if h1 else fallback_title
    if h1:
        h1.decompose()
    body_clean = clean_html(str(soup))
    return title_en, body_clean

import re

def clean_html(html):
    """
    Clean translated HTML before sending to WordPress:
    - Remove all <img> tags entirely
    - Strip all <a> hyperlinks but keep the anchor text
    """
    soup = BeautifulSoup(html, "html.parser")

    # Remove all images completely
    for img in soup.find_all("img"):
        img.decompose()

    # Strip links but keep their text
    for a in soup.find_all("a"):
        a.unwrap()  # removes the <a> tag, keeps inner text/HTML

    return str(soup)

def save_to_drive(title_en, body_en, original_url):
    plain_text = (
        f"Original DE URL: {original_url}\n{'=' * 60}\n\n"
        + BeautifulSoup(body_en, "html.parser").get_text(separator="\n\n")
    )
    safe_title = "".join(c for c in title_en if c.isalnum() or c in " -_")[:60].strip()
    filename   = f"{DRIVE_FOLDER}/[DRAFT] {safe_title}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(plain_text)
    return filename

import xmlrpc.client

def create_wp_draft(title_en, content_html, slug_en, lang="de_en"):
    resp = requests.post(
        f"{WP_URL}/wp-json/wp/v2/posts",
        headers=get_wp_headers(),
        json={
            "title": title_en,
            "content": content_html,
            "status": "draft",
            "slug": slug_en,
            "lang": lang
        },
        timeout=20
    )
    resp.raise_for_status()
    return resp.json()


    # Verify language was set
    verify = requests.get(
        f"{WP_URL}/wp-json/wp/v2/posts/{post['id']}",
        headers=get_wp_headers(),
        params={"lang": lang},
        timeout=10
    )
    print(f"  Post link: {post.get('link')}")
    return post

9- Full Pipeline

In [ ]:
def process_all_posts(n=10, min_clicks=50):
    all_posts          = get_top_german_blog_posts(gsc_service, GSC_SITE_URL)
    already_translated = load_translated_urls()

    new_posts = [
        p for p in all_posts
        if p["url"] not in already_translated and p["clicks"] >= min_clicks
    ][:n]

    print(f"📋 Already translated: {len(already_translated)}")
    print(f"🆕 Eligible this run:  {len(new_posts)}\n")

    if not new_posts:
        print("✅ Nothing new to translate — all top posts already done!")
        return []

    results = []
    for i, post in enumerate(new_posts, 1):
        url = post["url"]
        print(f"[{i}/{len(new_posts)}] {url}")
        try:
            title_de, html_de         = fetch_article_html(url)
            translated_html           = translate_article_html(title_de, html_de)
            title_en, body_en         = extract_title_and_body(translated_html, title_de)
            saved                     = save_to_drive(title_en, body_en, url)
            slug_en                   = urlparse(url).path.rstrip("/").split("/")[-1] + "-en"
            wp_post                   = create_wp_draft(title_en, body_en, slug_en)
            wp_id, wp_link            = wp_post.get("id"), wp_post.get("link")

            save_to_log(url, title_en, saved, post["clicks"], post["impressions"], post["position"], wp_id)

            print(f"  ✅ Drive → {saved}")
            print(f"  ✅ WP Draft #{wp_id}: {wp_link}")
            results.append({"de_url": url, "en_title": title_en, "wp_id": wp_id, "wp_link": wp_link})

        except Exception as e:
            print(f"  ❌ Skipped (will retry next run): {e}")

    print(f"\n🎉 Done! {len(results)}/{len(new_posts)} translated.")
    print(f"📋 Total in log: {len(load_translated_urls())}")
    return results

results = process_all_posts(n=10, min_clicks=50)

📋 Already translated: 22
🆕 Eligible this run:  10

[1/10] https://www.helpling.de/blog/stark-verschmutzte-dusche-reinigen-8-tipps/
  ✅ Drive → /content/drive/MyDrive/EN Blog Drafts/[DRAFT] Cleaning a Seriously Dirty Shower 8 Tips.txt
  ✅ WP Draft #15438: https://www.helpling.de/de_en/?p=15438
[2/10] https://www.helpling.de/blog/kuehlschrank-stinkt-trotz-reinigung/
  ✅ Drive → /content/drive/MyDrive/EN Blog Drafts/[DRAFT] Is Your Fridge Smelling Bad Despite Cleaning  Helpling DE.txt
  ✅ WP Draft #15439: https://www.helpling.de/de_en/?p=15439
[3/10] https://www.helpling.de/blog/lederjacke-waschen-die-besten-tipps-fuer-reinigung/
  ✅ Drive → /content/drive/MyDrive/EN Blog Drafts/[DRAFT] Washing Your Leather Jacket Top Tips  Helpling Blog EN.txt
  ✅ WP Draft #15440: https://www.helpling.de/de_en/?p=15440
[4/10] https://www.helpling.de/blog/wollteppich-reinigen-mit-diesen-3-methoden/
  ✅ Drive → /content/drive/MyDrive/EN Blog Drafts/[DRAFT] How to Clean Your Wool Carpet 3 Helpful Methods.tx

10- Log View

In [ ]:
view_log()


NameError: name 'view_log' is not defined